In [ ]:

import barnaba as bb
import numpy as np
import mdtraj as md
import matplotlib.pyplot as plt

# -----------------------------
# File paths
# -----------------------------
top = "./U_300/npt.gro"
traj = "./U_300/md_center.xtc"

# -----------------------------
# Load trajectory (for time)
# -----------------------------
t = md.load(traj, top=top)
time = t.time  # in ps

# -----------------------------
# Calculate eRMSD
# -----------------------------
ermsd_vals = bb.ermsd(reference=top,
                      target=traj,
                      topology=top)

# -----------------------------
# Print first 10 values
# -----------------------------
print("Frame\tTime (ps)\teRMSD")
for i in range(10):
    print(f"{i}\t{time[i]:.2f}\t{ermsd_vals[i]:.4f}")

# -----------------------------
# Save to file
# -----------------------------
data = np.column_stack((time, ermsd_vals))
np.savetxt("ermsd_vs_time.dat", data,
           header="Time(ps) eRMSD",
           fmt="%.6f")

print("\nSaved to ermsd_vs_time.dat")

# -----------------------------
# Plot
# -----------------------------
plt.figure()
plt.plot(time, ermsd_vals)
plt.xlabel("Time (ps)")
plt.ylabel("eRMSD")
plt.title("eRMSD vs Time")
plt.tight_layout()
plt.savefig("ermsd_plot.png", dpi=300)
plt.show()

In [2]:
import MDAnalysis as mda
from MDAnalysis.analysis.dihedrals import Dihedral
import numpy as np
import os

# Load topology and trajectory
u = mda.Universe("./U_300/npt.gro", "./U_300/md_center.xtc")

# Define backbone dihedral atoms
backbone = {
    "alpha": ["O3'", "P", "O5'", "C5'"],
    "beta":  ["P", "O5'", "C5'", "C4'"],
    "gamma": ["O5'", "C5'", "C4'", "C3'"],
    "delta": ["C5'", "C4'", "C3'", "O3'"],
    "epsilon": ["C4'", "C3'", "O3'", "P"],
    "zeta": ["C3'", "O3'", "P", "O5'"]
}

# Ensure output folder exists
os.makedirs("dihedrals_dat", exist_ok=True)

# Compute and save backbone dihedrals
for name, atoms in backbone.items():
    groups = []
    for res in u.residues:
        sel = res.atoms.select_atoms("name " + " ".join(atoms))
        if len(sel) == 4:
            groups.append(sel)
    dih = Dihedral(groups)
    dih.run()
    np.savetxt(f"dihedrals_dat/{name}.dat", dih.angles)
    print(f"Saved {name}.dat: {dih.angles.shape}")

# Compute and save glycosidic χ dihedral
chi_groups = []
for res in u.residues:
    base = res.resname.strip()
    if base in ["A", "G"]:  # purines
        atoms = ["O4'", "C1'", "N9", "C4"]
    else:  # pyrimidines
        atoms = ["O4'", "C1'", "N1", "C2"]
    sel = res.atoms.select_atoms("name " + " ".join(atoms))
    if len(sel) == 4:
        chi_groups.append(sel)
-
dih_chi = Dihedral(chi_groups)
dih_chi.run()
np.savetxt("dihedrals_dat/chi.dat", dih_chi.angles)
print(f"Saved chi.dat: {dih_chi.angles.shape}")

SyntaxError: invalid syntax (<ipython-input-2-cfdefdabd3fe>, line 45)

In [7]:
import MDAnalysis as mda
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
import numpy as np
import matplotlib.pyplot as plt
import os

# ===============================
# Parameters & Thresholds
# ===============================
DT_PS = 10.0               # Time between frames in XTC
FRAME_STRIDE = 5           # Analysis stride
STABILITY_THRESHOLD_PS = 2000.0  
MIN_HBONDS = 2             # Looking for > 2 H-bonds

REQUIRED_POINTS = int(STABILITY_THRESHOLD_PS / (DT_PS * FRAME_STRIDE))

DA_CUTOFF = 3.5  # Restored to standard; 2.8 is very strict for NWC
ANGLE_CUTOFF = 130
ROW_HEIGHT = 0.4
WC_TYPES = {frozenset(["RA", "RU"]), frozenset(["RG", "RC"])}

systems = {
    "1.0": ("./U_300/npt.gro", "./U_300/md_center.xtc"),
    "2.0": ("./U_2/npt.gro", "./U_2/md_center.xtc"),
    "3.0": ("./U_3/npt.gro", "./U_3/md_center.xtc"),
    "6.0": ("./U_6.0/npt.gro", "./U_6.0/md_center.xtc"),
    "8.0": ("./U_8.0/npt.gro", "./U_8.0/md_center.xtc"),
}

def normalize_base(resname):
    return resname.strip()[-1]

# ============================================================
# MAIN LOOP
# ============================================================
for label, (top, traj) in systems.items():
    if not os.path.exists(top) or not os.path.exists(traj):
        continue

    print(f"▶ Processing {label}M...")
    
    u = mda.Universe(top, traj)
    n_frames = len(u.trajectory[::FRAME_STRIDE])
    
    h = HydrogenBondAnalysis(
        universe=u,
        donors_sel="nucleic and (name N* or name O*)",
        hydrogens_sel="nucleic and name H*",
        acceptors_sel="nucleic and (name N* or name O*)",
        d_a_cutoff=DA_CUTOFF,
        d_h_a_angle_cutoff=ANGLE_CUTOFF,
        update_selections=False
    )
    h.run(step=FRAME_STRIDE)

    nwc_counts = {}
    for hb in h.results.hbonds:
        f = int(hb[0]) // FRAME_STRIDE
        if f >= n_frames: continue
        r1, r2 = u.atoms[int(hb[1])].residue, u.atoms[int(hb[3])].residue
        if r1.resid == r2.resid: continue
        b1, b2 = normalize_base(r1.resname), normalize_base(r2.resname)
        pair_key = tuple(sorted([(r1.resid, b1), (r2.resid, b2)]))
        
        if frozenset([b1, b2]) not in WC_TYPES:
            if pair_key not in nwc_counts:
                nwc_counts[pair_key] = np.zeros(n_frames, dtype=np.uint8)
            nwc_counts[pair_key][f] += 1

    # Filter for Stable & Strong
    stable_strong_keys = [
        p for p, counts in nwc_counts.items() 
        if (np.count_nonzero(counts) >= REQUIRED_POINTS) and (counts.max() > MIN_HBONDS)
    ]

    if not stable_strong_keys:
        print(f"  - No pairs met criteria for {label} M.")
        continue

    # Sort by occupancy (probability)
    stable_strong_keys = sorted(stable_strong_keys, 
                               key=lambda x: np.count_nonzero(nwc_counts[x])/n_frames, 
                               reverse=True)

    # --- DATA EXPORT (.dat file) ---
    dat_filename = f"NWC_stats_{label}M.dat"
    with open(dat_filename, "w") as f_out:
        f_out.write(f"# Residue_Pair\tProbability\tMean_HB\tMax_HB\n")
        for p in stable_strong_keys:
            counts = nwc_counts[p]
            prob = np.count_nonzero(counts) / n_frames
            avg_hb = counts.mean()
            max_hb = counts.max()
            pair_str = f"{p[0][0]}{p[0][1]}-{p[1][0]}{p[1][1]}"
            f_out.write(f"{pair_str}\t{prob:.4f}\t{avg_hb:.4f}\t{max_hb}\n")

    # --- PROBABILITY BAR GRAPH ---
    labels = [f"{p[0][0]}{p[0][1]}–{p[1][0]}{p[1][1]}" for p in stable_strong_keys]
    probs = [np.count_nonzero(nwc_counts[p]) / n_frames for p in stable_strong_keys]

    plt.figure(figsize=(10, max(4, len(labels) * 0.4)))
    plt.barh(labels, probs, color='teal', edgecolor='black')
    plt.axvline(x=REQUIRED_POINTS/n_frames, color='red', linestyle='--', label='Threshold')
    plt.xlabel("Occupancy Probability (Fraction of time)")
    plt.ylabel("Non-WC Base Pair")
    plt.title(f"Non-WC Pairing Probability (Urea {label} M)")
    plt.xlim(0, 1.0)
    plt.tight_layout()
    plt.savefig(f"NWC_probability_{label}M.png", dpi=300)
    plt.close()

    # --- TIME-EVOLUTION HEATMAP ---
    data_matrix = np.vstack([nwc_counts[p] for p in stable_strong_keys])
    time_axis = np.arange(n_frames) * (DT_PS * FRAME_STRIDE) / 1000.0 
    plt.figure(figsize=(12, max(4, len(stable_strong_keys) * ROW_HEIGHT)))
    im = plt.imshow(data_matrix, aspect="auto", origin="lower", cmap="magma", 
                    extent=[time_axis[0], time_axis[-1], -0.5, len(stable_strong_keys)-0.5],
                    vmin=0, vmax=4)
    plt.colorbar(im, label="H-bond Count")
    plt.yticks(np.arange(len(stable_strong_keys)), labels)
    plt.xlabel("Time (ns)")
    plt.title(f"Non-WC Persistence (>2 HB & >{STABILITY_THRESHOLD_PS} ps)\n(Urea {label} M)")
    plt.tight_layout()
    plt.savefig(f"NWC_heatmap_{label}M.png", dpi=300)
    plt.close()

print("\n✅ Analysis complete. Heatmaps, Probability graphs, and .dat files generated.")

▶ Processing 1.0M...


AttributeError: 'HydrogenBondAnalysis' object has no attribute 'results'